<a href="https://colab.research.google.com/github/andriakobaidze/ml-assn-04/blob/main/expression-recognition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import sys
import torch
import torch.nn as nn

!pip install wandb -q

import wandb
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: akoba23 (akoba23-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
df = pd.read_csv('/content/drive/MyDrive/ml-assn-04/train.csv')
print(df.head())
print(df.shape)
print(df.columns.tolist())

   emotion                                             pixels
0        0  70 80 82 72 58 58 60 63 54 58 60 48 89 115 121...
1        0  151 150 147 155 148 133 111 140 170 174 182 15...
2        2  231 212 156 164 174 138 161 173 182 200 106 38...
3        4  24 32 36 30 32 23 19 20 30 41 21 22 32 34 21 1...
4        6  4 0 0 0 0 0 0 0 0 0 0 0 3 15 23 28 48 50 58 84...
(28709, 2)
['emotion', 'pixels']


In [5]:
!git clone https://github.com/andriakobaidze/ml-assn-04.git
%cd ml-assn-04

Cloning into 'ml-assn-04'...
remote: Enumerating objects: 43, done.
remote: Counting objects: 100% (43/43), done.
remote: Compressing objects: 100% (31/31), done.
remote: Total 43 (delta 22), reused 26 (delta 11), pack-reused 0 (from 0)
Receiving objects: 100% (43/43), 12.84 KiB | 12.84 MiB/s, done.
Resolving deltas: 100% (22/22), done.
/content/ml-assn-04


In [6]:
!pip install torch torchvision wandb numpy pandas matplotlib scikit-learn Pillow tqdm -q

In [7]:
sys.path.append('/content/ml-assn-04')

from dataset import get_dataloaders

train_loader, val_loader, test_loader = get_dataloaders(
    train_csv_path='/content/drive/MyDrive/ml-assn-04/train.csv',
    test_csv_path='/content/drive/MyDrive/ml-assn-04/test.csv',
    batch_size=64
)

images, labels = next(iter(train_loader))
print(images.shape)
print(labels.shape)

Train size: 22968
Val size:   5741
Test size:  7178
torch.Size([64, 1, 48, 48])
torch.Size([64])


In [25]:
sys.path.append('/content/ml-assn-04')

from models import SimpleCNN
from trainer import train

config = {
    'architecture': 'SimpleCNN',
    'epochs': 10,
    'lr': 0.001,
    'batch_size': 64,
    'optimizer': 'Adam',
    'scheduler': 'ReduceLROnPlateau',
}

run = wandb.init(
    project="ML-assn-04",
    name="SimpleCNN-baseline",
    config=config
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

model = SimpleCNN(num_classes=7).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=config['lr'])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)

best_model_path = train(
    model, train_loader, val_loader,
    criterion, optimizer, scheduler,
    device, config
)

run.finish()

Using device: cuda


Epoch 1 [Val]: 100%|██████████| 90/90 [00:03<00:00, 22.52it/s, acc=32.6, loss=1.64]


Epoch 1/10 | Train Loss: 1.7778, Train Acc: 26.88% | Val Loss: 1.7071, Val Acc: 32.57%
Best model saved (val_acc: 32.57%)


Epoch 2 [Val]: 100%|██████████| 90/90 [00:04<00:00, 21.62it/s, acc=36.8, loss=1.62]


Epoch 2/10 | Train Loss: 1.7148, Train Acc: 30.84% | Val Loss: 1.6346, Val Acc: 36.77%
Best model saved (val_acc: 36.77%)


...

Epoch 9 [Val]: 100%|██████████| 90/90 [00:02<00:00, 31.10it/s, acc=48, loss=1.29]


Epoch 9/10 | Train Loss: 1.4798, Train Acc: 42.63% | Val Loss: 1.3683, Val Acc: 48.02%
Best model saved (val_acc: 48.02%)


Epoch 10 [Val]: 100%|██████████| 90/90 [00:03<00:00, 24.42it/s, acc=49.5, loss=1.24]

Epoch 10/10 | Train Loss: 1.4598, Train Acc: 43.64% | Val Loss: 1.3519, Val Acc: 49.50%
Best model saved (val_acc: 49.50%)

Training complete. Best Val Acc: 49.50%


batch,▁▁▁▁▁▂▂▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇█
batch_loss,▆▇█▇▇▆▆▇▆▅▆▅▆▆▇▅▇▅▄▄▄▅▃▄▆▂▃▃▄▃▃▄▃▂▃▃▃▁▃▃
best_val_acc,▁
epoch,▁▂▃▃▄▅▆▆▇█
lr,▁▁▁▁▁▁▁▁▁▁
train_acc,▁▃▄▅▆▆▇▇██
train_loss,█▇▅▅▄▃▂▂▁▁
val_acc,▁▃▅▄▆▆▇▇▇█
val_loss,█▇▅▅▃▃▂▁▁▁
batch,3948
batch_loss,1.5328


In [26]:
import importlib
import models, trainer
importlib.reload(models)
importlib.reload(trainer)

from models import MediumCNN
from trainer import train

config = {
    'architecture': 'MediumCNN',
    'epochs': 15,
    'lr': 0.001,
    'batch_size': 64,
    'optimizer': 'Adam',
    'scheduler': 'ReduceLROnPlateau',
}

run = wandb.init(
    project="ML-assn-04",
    name="MediumCNN-baseline",
    config=config
)

model = MediumCNN(num_classes=7).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=config['lr'])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)

best_model_path = train(
    model, train_loader, val_loader,
    criterion, optimizer, scheduler,
    device, config
)

run.finish()

Epoch 1 [Val]: 100%|██████████| 90/90 [00:03<00:00, 29.92it/s, acc=40.7, loss=1.46]


Epoch 1/15 | Train Loss: 1.7578, Train Acc: 29.29% | Val Loss: 1.5303, Val Acc: 40.71%
Best model saved (val_acc: 40.71%)


Epoch 2 [Val]: 100%|██████████| 90/90 [00:02<00:00, 30.46it/s, acc=45.6, loss=1.4]


Epoch 2/15 | Train Loss: 1.5813, Train Acc: 37.85% | Val Loss: 1.4340, Val Acc: 45.57%
Best model saved (val_acc: 45.57%)


Epoch 3 [Val]: 100%|██████████| 90/90 [00:02<00:00, 30.33it/s, acc=47.7, loss=1.3]


...

Epoch 14 [Val]: 100%|██████████| 90/90 [00:02<00:00, 30.45it/s, acc=55.8, loss=1.1]


Epoch 14/15 | Train Loss: 1.3126, Train Acc: 49.53% | Val Loss: 1.1556, Val Acc: 55.79%
Best model saved (val_acc: 55.79%)


Epoch 15 [Val]: 100%|██████████| 90/90 [00:03<00:00, 29.86it/s, acc=57.2, loss=1.22]

Epoch 15/15 | Train Loss: 1.2957, Train Acc: 50.40% | Val Loss: 1.1427, Val Acc: 57.17%
Best model saved (val_acc: 57.17%)

Training complete. Best Val Acc: 57.17%


batch,▁▁▁▂▂▂▂▂▂▂▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇████
batch_loss,█▃▂▂▂▂▂▂▂▂▁▁▁▁▂▂▂▂▂▂▂▁▁▂▁▂▂▁▂▁▂▂▂▂▁▂▁▁▁▁
best_val_acc,▁
epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
lr,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_acc,▁▄▅▅▆▆▇▇▇▇▇▇███
train_loss,█▅▄▄▃▃▃▂▂▂▂▂▁▁▁
val_acc,▁▃▄▄▅▆▅▆▆▇▆▇▇▇█
val_loss,█▆▅▄▃▃▄▃▃▂▃▃▂▁▁
batch,5743
batch_loss,1.12962


In [33]:
from models import DeepCNN
from trainer import train

config = {
    'architecture': 'DeepCNN',
    'epochs': 20,
    'lr': 0.001,
    'batch_size': 64,
    'optimizer': 'Adam',
    'scheduler': 'ReduceLROnPlateau',
}

run = wandb.init(
    project="ML-assn-04",
    name="DeepCNN-baseline",
    config=config
)

model = DeepCNN(num_classes=7).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=config['lr'])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)

best_model_path = train(
    model, train_loader, val_loader,
    criterion, optimizer, scheduler,
    device, config
)

run.finish()

Epoch 1 [Val]: 100%|██████████| 90/90 [00:02<00:00, 30.01it/s, acc=35.6, loss=1.5]


Epoch 1/20 | Train Loss: 1.7267, Train Acc: 29.02% | Val Loss: 1.7159, Val Acc: 35.62%
Best model saved (val_acc: 35.62%)


Epoch 2 [Val]: 100%|██████████| 90/90 [00:03<00:00, 29.79it/s, acc=34.8, loss=1.68]


Epoch 2/20 | Train Loss: 1.5193, Train Acc: 41.06% | Val Loss: 1.8691, Val Acc: 34.77%


Epoch 3 [Val]: 100%|██████████| 90/90 [00:03<00:00, 29.47it/s, acc=50.5, loss=1.28]


...

Epoch 19 [Val]: 100%|██████████| 90/90 [00:03<00:00, 29.73it/s, acc=61.6, loss=1.18]


Epoch 19/20 | Train Loss: 1.1192, Train Acc: 57.63% | Val Loss: 1.0236, Val Acc: 61.63%
Best model saved (val_acc: 61.63%)


Epoch 20 [Val]: 100%|██████████| 90/90 [00:03<00:00, 28.85it/s, acc=60.4, loss=1.07]

Epoch 20/20 | Train Loss: 1.1012, Train Acc: 58.63% | Val Loss: 1.0425, Val Acc: 60.37%

Training complete. Best Val Acc: 61.63%


batch,▁▁▁▁▁▂▂▂▂▂▂▂▂▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▇▇▇▇▇█
batch_loss,▆▇▆▆█▅▅▅▅▆▆▄▄▅▄▄▄▄▃▄▄▄▃▄▄▄▂▃▃▄▂▃▃▂▂▄▂▃▁▃
best_val_acc,▁
epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
lr,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_acc,▁▄▅▅▆▆▆▆▇▇▇▇▇▇▇█████
train_loss,█▆▅▄▃▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁
val_acc,▁▁▅▆▆▆▆▇▇▇▇▇▇▇██████
val_loss,▇█▃▃▃▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁
batch,7538
batch_loss,0.9087


In [8]:
for mod in ['models', 'trainer']:
    if mod in sys.modules:
        del sys.modules[mod]

from models import DeepCNN
from trainer import train

for lr in [0.0001, 0.01]:
    config = {
        'architecture': 'DeepCNN',
        'epochs': 15,
        'lr': lr,
        'batch_size': 64,
        'optimizer': 'Adam',
        'scheduler': 'ReduceLROnPlateau',
    }

    run = wandb.init(
        project="ML-assn-04",
        name=f"DeepCNN-lr{lr}",
        config=config
    )

    model = DeepCNN(num_classes=7).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)

    best_model_path = train(
        model, train_loader, val_loader,
        criterion, optimizer, scheduler,
        device, config
    )

    run.finish()

NameError: name 'device' is not defined

In [16]:
for mod in ['models', 'trainer']:
    if mod in sys.modules:
        del sys.modules[mod]

from models import DeepCNN
from trainer import train

for lr in [0.0001, 0.01]:
    config = {
        'architecture': 'DeepCNN',
        'epochs': 15,
        'lr': lr,
        'batch_size': 64,
        'optimizer': 'Adam',
        'scheduler': 'ReduceLROnPlateau',
    }

    run = wandb.init(
        project="ML-assn-04",
        name=f"DeepCNN-lr{lr}",
        config=config
    )

    model = DeepCNN(num_classes=7).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)

    best_model_path = train(
        model, train_loader, val_loader,
        criterion, optimizer, scheduler,
        device, config
    )

    run.finish()

Epoch 1 [Val]: 100%|██████████| 90/90 [00:03<00:00, 28.55it/s, acc=25.1, loss=1.74]


Epoch 1/15 | Train Loss: 1.7975, Train Acc: 25.18% | Val Loss: 1.8139, Val Acc: 25.13%
Best model saved (val_acc: 25.13%)


Epoch 2 [Val]: 100%|██████████| 90/90 [00:03<00:00, 28.48it/s, acc=34.7, loss=1.56]


Epoch 2/15 | Train Loss: 1.7309, Train Acc: 28.94% | Val Loss: 1.6571, Val Acc: 34.66%
Best model saved (val_acc: 34.66%)


Epoch 3 [Val]: 100%|██████████| 90/90 [00:03<00:00, 28.70it/s, acc=40.8, loss=1.44]


...

Epoch 14 [Val]: 100%|██████████| 90/90 [00:03<00:00, 27.67it/s, acc=25.2, loss=1.76]


Epoch 14/15 | Train Loss: 1.8106, Train Acc: 25.13% | Val Loss: 1.8069, Val Acc: 25.15%


Epoch 15 [Val]: 100%|██████████| 90/90 [00:03<00:00, 26.70it/s, acc=25.2, loss=1.76]


Epoch 15/15 | Train Loss: 1.8104, Train Acc: 25.13% | Val Loss: 1.8069, Val Acc: 25.15%

Training complete. Best Val Acc: 25.24%


batch,▁▁▁▂▂▂▂▂▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▅▅▅▆▆▆▆▇▇▇███████
batch_loss,▅▆▂▂▄▂▅▃▃▄▅█▃▂▃▄▅▄▃▃▄▅▄▄▃▂▂▄▃▃▃▁▄▃▃▆▃▆▃▅
best_val_acc,▁
epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
lr,█████▄▄▄▄▂▂▂▂▁▁
train_acc,▁█▆████████████
train_loss,█▂▁▃▂▂▂▂▂▂▂▂▂▂▂
val_acc,▁█▁▁▁▁▁▁▁▁▁▁▁▁▁
val_loss,▄▁█▇▇▆▇▆▆▇▆▇▆▆▆
batch,5743
batch_loss,1.71733


In [11]:
sys.path.append('/content/ml-assn-04')

for mod in ['models', 'trainer', 'dataset']:
    if mod in sys.modules:
        del sys.modules[mod]

from dataset import get_dataloaders
from models import DeepCNN
from trainer import train

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

train_loader, val_loader, test_loader = get_dataloaders(
    train_csv_path='/content/drive/MyDrive/ml-assn-04/train.csv',
    test_csv_path='/content/drive/MyDrive/ml-assn-04/test.csv',
    batch_size=64
)

Using device: cuda
Train size: 22968
Val size:   5741
Test size:  7178


In [17]:
from models import DeepCNN
from trainer import train

config = {
    'architecture': 'DeepCNN',
    'epochs': 20,
    'lr': 0.01,
    'batch_size': 64,
    'optimizer': 'SGD',
    'scheduler': 'ReduceLROnPlateau',
    'momentum': 0.9,
    'weight_decay': 1e-4,
}

run = wandb.init(
    project="ML-assn-04",
    name="DeepCNN-SGD",
    config=config
)

model = DeepCNN(num_classes=7).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(
    model.parameters(),
    lr=config['lr'],
    momentum=config['momentum'],
    weight_decay=config['weight_decay']
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)

best_model_path = train(
    model, train_loader, val_loader,
    criterion, optimizer, scheduler,
    device, config
)

run.finish()

Epoch 1 [Val]: 100%|██████████| 90/90 [00:04<00:00, 22.36it/s, acc=25.4, loss=1.74]


Epoch 1/20 | Train Loss: 1.7905, Train Acc: 25.09% | Val Loss: 1.8405, Val Acc: 25.38%
Best model saved (val_acc: 25.38%)


Epoch 2 [Val]: 100%|██████████| 90/90 [00:03<00:00, 28.31it/s, acc=32, loss=1.52]


Epoch 2/20 | Train Loss: 1.7000, Train Acc: 30.60% | Val Loss: 1.6980, Val Acc: 32.02%
Best model saved (val_acc: 32.02%)


Epoch 3 [Val]: 100%|██████████| 90/90 [00:03<00:00, 27.98it/s, acc=42.5, loss=1.41]


...

Epoch 19 [Val]: 100%|██████████| 90/90 [00:05<00:00, 17.27it/s, acc=60.4, loss=1.14]


Epoch 19/20 | Train Loss: 1.1668, Train Acc: 55.49% | Val Loss: 1.0550, Val Acc: 60.39%
Best model saved (val_acc: 60.39%)


Epoch 20 [Val]: 100%|██████████| 90/90 [00:04<00:00, 18.45it/s, acc=56.3, loss=1.19]

Epoch 20/20 | Train Loss: 1.1529, Train Acc: 56.09% | Val Loss: 1.1104, Val Acc: 56.28%

Training complete. Best Val Acc: 60.39%


batch,▁▁▁▁▁▂▂▂▂▂▂▂▂▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▇▇▇███
batch_loss,███▇▇▇▇▇▅▄▃▅▄▅▄▅▃▆▃▅▅▄▃▃▃▄▃▃▃▁▃▃▁▄▃▂▂▂▄▃
best_val_acc,▁
epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
lr,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_acc,▁▂▄▅▅▆▆▆▇▇▇▇▇▇▇█████
train_loss,█▇▆▅▄▄▃▃▃▂▂▂▂▂▂▂▁▁▁▁
val_acc,▁▂▄▅▅▅▆▆▆▆▆▇▇▇▇▇▇██▇
val_loss,█▇▅▅▄▄▃▃▃▃▃▂▂▂▂▂▂▁▁▁
batch,7538
batch_loss,1.12957


In [19]:
for mod in ['models', 'trainer']:
    if mod in sys.modules:
        del sys.modules[mod]

from models import ResNet18FER
from trainer import train

config = {
    'architecture': 'ResNet18FER',
    'epochs': 20,
    'lr': 0.001,
    'batch_size': 64,
    'optimizer': 'Adam',
    'scheduler': 'ReduceLROnPlateau',
}

run = wandb.init(
    project="ML-assn-04",
    name="ResNet18-pretrained",
    config=config
)

model = ResNet18FER(num_classes=7).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=config['lr'])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)

best_model_path = train(
    model, train_loader, val_loader,
    criterion, optimizer, scheduler,
    device, config
)

run.finish()

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 225MB/s]
Epoch 1 [Val]: 100%|██████████| 90/90 [00:03<00:00, 26.95it/s, acc=38.3, loss=1.58]


Epoch 1/20 | Train Loss: 1.7914, Train Acc: 32.04% | Val Loss: 1.5964, Val Acc: 38.30%
Best model saved (val_acc: 38.30%)


Epoch 2 [Val]: 100%|██████████| 90/90 [00:03<00:00, 24.03it/s, acc=44.5, loss=1.53]


Epoch 2/20 | Train Loss: 1.5611, Train Acc: 39.64% | Val Loss: 1.4503, Val Acc: 44.54%
Best model saved (val_acc: 44.54%)


...

Epoch 19 [Val]: 100%|██████████| 90/90 [00:03<00:00, 26.22it/s, acc=57.5, loss=1.16]


Epoch 19/20 | Train Loss: 1.0801, Train Acc: 59.05% | Val Loss: 1.1170, Val Acc: 57.52%


Epoch 20 [Val]: 100%|██████████| 90/90 [00:03<00:00, 27.39it/s, acc=46.3, loss=1.62]


Epoch 20/20 | Train Loss: 1.0701, Train Acc: 59.96% | Val Loss: 1.4359, Val Acc: 46.32%

Training complete. Best Val Acc: 59.38%


batch,▁▁▁▁▁▂▂▂▂▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▇▇▇▇▇█
batch_loss,█▇▇█▅▇▅▅▄▄▆▅▅▄▇▄▅▄▅▃▄▄▅▅▄▂▆▂▁▂▃▁▂▃▃▁▄▆▄▃
best_val_acc,▁
epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
lr,███████████████████▁
train_acc,▁▃▄▅▅▅▅▆▆▅▆▇▇▇▇▇▇███
train_loss,█▆▅▄▄▄▄▃▃▄▃▂▂▂▂▂▂▁▁▁
val_acc,▂▄▅▅▃▃▆▆▁▇▆█▇▇▇██▇▇▄
val_loss,▇▅▄▃▆▆▂▃█▂▃▁▂▁▂▁▁▂▁▅
batch,7538
batch_loss,1.41042


In [20]:
for mod in ['models', 'trainer']:
    if mod in sys.modules:
        del sys.modules[mod]

from models import ResNet18FER
from trainer import train

config = {
    'architecture': 'ResNet18FER',
    'epochs': 5,
    'lr': 0.001,
    'batch_size': 64,
    'optimizer': 'Adam',
    'scheduler': 'ReduceLROnPlateau',
    'stage': 'frozen-backbone'
}

run = wandb.init(
    project="ML-assn-04",
    name="ResNet18-stage1-frozen",
    config=config
)

model = ResNet18FER(num_classes=7).to(device)

for name, param in model.named_parameters():
    if 'fc' not in name:
        param.requires_grad = False

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=config['lr']
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2, factor=0.5)

best_model_path = train(
    model, train_loader, val_loader,
    criterion, optimizer, scheduler,
    device, config
)

run.finish()

Epoch 1 [Val]: 100%|██████████| 90/90 [00:04<00:00, 20.87it/s, acc=26.7, loss=1.79]


Epoch 1/5 | Train Loss: 1.9819, Train Acc: 21.60% | Val Loss: 1.7963, Val Acc: 26.65%
Best model saved (val_acc: 26.65%)


Epoch 2 [Val]: 100%|██████████| 90/90 [00:03<00:00, 26.60it/s, acc=24.8, loss=1.71]


Epoch 2/5 | Train Loss: 1.8591, Train Acc: 23.57% | Val Loss: 1.7969, Val Acc: 24.80%


Epoch 3 [Val]: 100%|██████████| 90/90 [00:03<00:00, 27.43it/s, acc=26.4, loss=1.78]


...

Epoch 4 [Val]: 100%|██████████| 90/90 [00:03<00:00, 27.20it/s, acc=26.9, loss=1.79]


Epoch 4/5 | Train Loss: 1.8557, Train Acc: 23.69% | Val Loss: 1.7739, Val Acc: 26.95%
Best model saved (val_acc: 26.95%)


Epoch 5 [Val]: 100%|██████████| 90/90 [00:03<00:00, 26.70it/s, acc=26.8, loss=1.8]

Epoch 5/5 | Train Loss: 1.8500, Train Acc: 23.95% | Val Loss: 1.8164, Val Acc: 26.82%

Training complete. Best Val Acc: 26.95%


batch,▁▁▂▂▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▆▆▆▆▆▆▆▆▆▇▇▇▇██
batch_loss,▄█▅▆▅▃▃▂▃▇▃▃▂▃▃▄▄▃▄▃▅▃▃▂▃▃▃▂▁▃▄▃▄▃▄▃▄▂▂▂
best_val_acc,▁
epoch,▁▃▅▆█
lr,▁▁▁▁▁
train_acc,▁▇█▇█
train_loss,█▁▁▁▁
val_acc,▇▁▆██
val_loss,▅▅▅▁█
batch,2153
batch_loss,1.77477


In [23]:
for mod in ['models', 'trainer']:
    if mod in sys.modules:
        del sys.modules[mod]

from models import CNNLSTM
from trainer import train

config = {
    'architecture': 'CNNLSTM',
    'epochs': 20,
    'lr': 0.001,
    'batch_size': 64,
    'optimizer': 'Adam',
    'scheduler': 'ReduceLROnPlateau',
    'hidden_size': 256,
    'num_layers': 2,
}

run = wandb.init(
    project="ML-assn-04",
    name="CNNLSTM-baseline",
    config=config
)

model = CNNLSTM(num_classes=7, hidden_size=256, num_layers=2).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=config['lr'])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)

best_model_path = train(
    model, train_loader, val_loader,
    criterion, optimizer, scheduler,
    device, config
)

run.finish()

Epoch 1 [Val]: 100%|██████████| 90/90 [00:03<00:00, 28.43it/s, acc=33.1, loss=1.51]


Epoch 1/20 | Train Loss: 1.7492, Train Acc: 27.43% | Val Loss: 1.6839, Val Acc: 33.10%
Best model saved (val_acc: 33.10%)


Epoch 2 [Val]: 100%|██████████| 90/90 [00:03<00:00, 28.93it/s, acc=43.8, loss=1.35]


Epoch 2/20 | Train Loss: 1.5487, Train Acc: 38.54% | Val Loss: 1.4459, Val Acc: 43.76%
Best model saved (val_acc: 43.76%)


Epoch 3 [Val]: 100%|██████████| 90/90 [00:03<00:00, 28.97it/s, acc=47.6, loss=1.36]


...

Epoch 19 [Val]: 100%|██████████| 90/90 [00:04<00:00, 19.45it/s, acc=59.3, loss=1.1]


Epoch 19/20 | Train Loss: 1.0875, Train Acc: 58.47% | Val Loss: 1.0672, Val Acc: 59.26%


Epoch 20 [Val]: 100%|██████████| 90/90 [00:05<00:00, 17.15it/s, acc=60.3, loss=1.07]

Epoch 20/20 | Train Loss: 1.0637, Train Acc: 59.56% | Val Loss: 1.0358, Val Acc: 60.30%

Training complete. Best Val Acc: 60.58%


batch,▁▁▁▂▂▂▃▃▃▃▃▃▃▃▄▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▇▇▇▇▇█████
batch_loss,▇█▅█▅▆▅▆▆▆▅▅▆▅▆▅▄▄▃▃▄▅▃▂▄▃▂▄▄▄▅▃▂▁▃▄▂▄▂▃
best_val_acc,▁
epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
lr,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_acc,▁▃▄▅▅▆▆▆▇▇▇▇▇▇▇█████
train_loss,█▆▅▄▄▃▃▃▃▂▂▂▂▂▂▂▁▁▁▁
val_acc,▁▄▅▅▆▆▆▆▆▇▇▇▇▇▇▇████
val_loss,█▅▄▄▃▃▃▃▃▂▂▂▂▂▂▂▁▁▁▁
batch,7538
batch_loss,1.11827


In [25]:
for mod in ['models', 'trainer']:
    if mod in sys.modules:
        del sys.modules[mod]

from models import CNNLSTM
from trainer import train

config = {
    'architecture': 'CNNLSTM',
    'epochs': 20,
    'lr': 0.001,
    'batch_size': 64,
    'optimizer': 'Adam',
    'scheduler': 'ReduceLROnPlateau',
    'hidden_size': 512,
    'num_layers': 2,
}

run = wandb.init(
    project="ML-assn-04",
    name="CNNLSTM-hidden512",
    config=config
)

model = CNNLSTM(num_classes=7, hidden_size=512, num_layers=2).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=config['lr'])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)

best_model_path = train(
    model, train_loader, val_loader,
    criterion, optimizer, scheduler,
    device, config
)

run.finish()

Epoch 1 [Val]: 100%|██████████| 90/90 [00:03<00:00, 25.35it/s, acc=36.4, loss=1.52]


Epoch 1/20 | Train Loss: 1.7414, Train Acc: 28.44% | Val Loss: 1.5916, Val Acc: 36.44%
Best model saved (val_acc: 36.44%)


Epoch 2 [Val]: 100%|██████████| 90/90 [00:03<00:00, 23.07it/s, acc=43, loss=1.37]


Epoch 2/20 | Train Loss: 1.5424, Train Acc: 38.62% | Val Loss: 1.4594, Val Acc: 43.04%
Best model saved (val_acc: 43.04%)


Epoch 3 [Val]: 100%|██████████| 90/90 [00:03<00:00, 28.71it/s, acc=44.5, loss=1.47]


...

Epoch 19 [Val]: 100%|██████████| 90/90 [00:03<00:00, 28.41it/s, acc=57.8, loss=1.22]


Epoch 19/20 | Train Loss: 1.0625, Train Acc: 59.60% | Val Loss: 1.0798, Val Acc: 57.76%


Epoch 20 [Val]: 100%|██████████| 90/90 [00:03<00:00, 28.67it/s, acc=58.5, loss=1.3]


Epoch 20/20 | Train Loss: 1.0533, Train Acc: 59.81% | Val Loss: 1.1007, Val Acc: 58.51%

Training complete. Best Val Acc: 58.53%


batch,▁▁▁▁▁▂▂▂▂▃▃▃▃▄▄▄▅▅▅▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇▇██
batch_loss,█▇▆▆▄▆▅▅▆█▄▄▄▄▄▅▂▄▄▄▅▃▃▂▅▂▃▂▃▃▃▂▂▃▃▄▁▁▁▁
best_val_acc,▁
epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
lr,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_acc,▁▃▄▅▅▅▆▆▆▇▇▇▇▇▇▇████
train_loss,█▆▅▅▄▄▃▃▃▃▂▂▂▂▂▂▁▁▁▁
val_acc,▁▃▄▄▆▅▆▆▇▇▇▇▇▇▇█▇███
val_loss,█▆▆▅▄▄▃▄▃▂▂▂▂▂▂▁▂▁▁▁
batch,7538
batch_loss,0.9996


In [26]:
for mod in ['models', 'trainer']:
    if mod in sys.modules:
        del sys.modules[mod]

from models import CNNLSTM
from trainer import train

config = {
    'architecture': 'CNNLSTM',
    'epochs': 20,
    'lr': 0.001,
    'batch_size': 64,
    'optimizer': 'Adam',
    'scheduler': 'ReduceLROnPlateau',
    'hidden_size': 256,
    'num_layers': 1,
}

run = wandb.init(
    project="ML-assn-04",
    name="CNNLSTM-1layer",
    config=config
)

model = CNNLSTM(num_classes=7, hidden_size=256, num_layers=1).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=config['lr'])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)

best_model_path = train(
    model, train_loader, val_loader,
    criterion, optimizer, scheduler,
    device, config
)

run.finish()

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/rnn.py:1013: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.3 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)
Epoch 1 [Val]: 100%|██████████| 90/90 [00:03<00:00, 29.44it/s, acc=40, loss=1.5]


Epoch 1/20 | Train Loss: 1.7215, Train Acc: 29.42% | Val Loss: 1.5276, Val Acc: 40.03%
Best model saved (val_acc: 40.03%)


Epoch 2 [Val]: 100%|██████████| 90/90 [00:03<00:00, 26.85it/s, acc=43.4, loss=1.41]


Epoch 2/20 | Train Loss: 1.5249, Train Acc: 40.25% | Val Loss: 1.4576, Val Acc: 43.42%
Best model saved (val_acc: 43.42%)


Epoch 3 [Val]: 100%|██████████| 90/90 [00:03<00:00, 28.92it/s, acc=46.6, loss=1.34]


...

Epoch 19 [Val]: 100%|██████████| 90/90 [00:03<00:00, 29.06it/s, acc=60.5, loss=1.09]


Epoch 19/20 | Train Loss: 1.0545, Train Acc: 59.88% | Val Loss: 1.0396, Val Acc: 60.46%
Best model saved (val_acc: 60.46%)


Epoch 20 [Val]: 100%|██████████| 90/90 [00:03<00:00, 28.86it/s, acc=61.3, loss=1.08]


Epoch 20/20 | Train Loss: 1.0503, Train Acc: 60.20% | Val Loss: 1.0193, Val Acc: 61.28%
Best model saved (val_acc: 61.28%)

Training complete. Best Val Acc: 61.28%


batch,▁▁▁▁▁▂▂▃▃▃▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇█
batch_loss,█▇▆▆▆▆▇▄▆▆▅▅▄▅▃▅▅▄▄▄▃▃▃▄▄▅▃▃▃▄▄▃▄▁▄▂▂▂▂▃
best_val_acc,▁
epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
lr,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_acc,▁▃▄▅▅▅▆▆▆▆▇▇▇▇▇█████
train_loss,█▆▅▅▄▄▃▃▃▃▂▂▂▂▂▁▁▁▁▁
val_acc,▁▂▃▃▄▅▅▆▆▆▆▆▆▇▇▇▇███
val_loss,█▇▆▅▅▄▄▄▃▃▂▃▂▂▂▂▂▁▁▁
batch,7538
batch_loss,1.02571


In [17]:
for mod in ['models', 'trainer']:
    if mod in sys.modules:
        del sys.modules[mod]

from models import CNNLSTM
from trainer import train

config = {
    'architecture': 'CNNLSTM',
    'epochs': 20,
    'lr': 0.001,
    'batch_size': 64,
    'optimizer': 'Adam',
    'scheduler': 'ReduceLROnPlateau',
    'hidden_size': 256,
    'num_layers': 2,
    'dropout': 0.5,
}

run = wandb.init(
    project="ML-assn-04",
    name="CNNLSTM-dropout0.5",
    config=config
)

model = CNNLSTM(num_classes=7, hidden_size=256, num_layers=2).to(device)

model.lstm = torch.nn.LSTM(
    input_size=128 * 6,
    hidden_size=256,
    num_layers=2,
    batch_first=True,
    dropout=0.5
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=config['lr'])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)

best_model_path = train(
    model, train_loader, val_loader,
    criterion, optimizer, scheduler,
    device, config
)

run.finish()

Epoch 1 [Val]: 100%|██████████| 90/90 [00:03<00:00, 28.18it/s, acc=23, loss=1.89]


Epoch 1/20 | Train Loss: 1.7685, Train Acc: 26.68% | Val Loss: 1.9237, Val Acc: 23.01%
Best model saved (val_acc: 23.01%)


Epoch 2 [Val]: 100%|██████████| 90/90 [00:03<00:00, 28.55it/s, acc=40.5, loss=1.44]


Epoch 2/20 | Train Loss: 1.5881, Train Acc: 36.70% | Val Loss: 1.5430, Val Acc: 40.48%
Best model saved (val_acc: 40.48%)


Epoch 3 [Val]: 100%|██████████| 90/90 [00:03<00:00, 28.55it/s, acc=45.1, loss=1.34]


...

Epoch 19 [Val]: 100%|██████████| 90/90 [00:03<00:00, 27.54it/s, acc=58.1, loss=1.24]


Epoch 19/20 | Train Loss: 1.0860, Train Acc: 58.56% | Val Loss: 1.1034, Val Acc: 58.14%


Epoch 20 [Val]: 100%|██████████| 90/90 [00:03<00:00, 29.63it/s, acc=60.5, loss=1.16]

Epoch 20/20 | Train Loss: 1.0776, Train Acc: 58.93% | Val Loss: 1.0535, Val Acc: 60.51%
Best model saved (val_acc: 60.51%)

Training complete. Best Val Acc: 60.51%


batch,▁▁▁▂▂▂▂▂▂▂▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▅▅▆▆▆▇▇▇▇▇▇███
batch_loss,▇█▇██▆▆▅▄▄▂▃▅▄▄▃▄▃▃▄▄▂▁▂▂▃▁▄▃▂▂▅▁▂▂▃▂▃▁▁
best_val_acc,▁
epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
lr,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_acc,▁▃▄▅▅▆▆▆▇▇▇▇▇▇▇█████
train_loss,█▆▅▄▄▄▃▃▃▃▂▂▂▂▂▂▁▁▁▁
val_acc,▁▄▅▅▆▆▇▇▇▇▇▇█▇▇█████
val_loss,█▅▄▄▃▃▂▂▂▂▂▂▁▂▂▁▁▁▁▁
batch,7538
batch_loss,1.09936


In [20]:
for mod in ['models', 'trainer']:
    if mod in sys.modules:
        del sys.modules[mod]

from models import ViTFER
from trainer import train

config = {
    'architecture': 'ViTFER',
    'epochs': 20,
    'lr': 0.001,
    'batch_size': 64,
    'optimizer': 'Adam',
    'scheduler': 'ReduceLROnPlateau',
    'embed_dim': 128,
    'num_heads': 4,
    'num_layers': 4,
    'patch_size': 8,
}

run = wandb.init(
    project="ML-assn-04",
    name="ViT-baseline",
    config=config
)

model = ViTFER(num_classes=7).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=config['lr'])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)

best_model_path = train(
    model, train_loader, val_loader,
    criterion, optimizer, scheduler,
    device, config
)

run.finish()

Epoch 1 [Val]: 100%|██████████| 90/90 [00:05<00:00, 16.82it/s, acc=25.8, loss=1.74]


Epoch 1/20 | Train Loss: 1.8237, Train Acc: 23.56% | Val Loss: 1.7930, Val Acc: 25.76%
Best model saved (val_acc: 25.76%)


Epoch 2 [Val]: 100%|██████████| 90/90 [00:03<00:00, 28.46it/s, acc=25.2, loss=1.72]


Epoch 2/20 | Train Loss: 1.8003, Train Acc: 24.72% | Val Loss: 1.7874, Val Acc: 25.15%


Epoch 3 [Val]: 100%|██████████| 90/90 [00:03<00:00, 29.24it/s, acc=25.2, loss=1.71]


...

Epoch 19 [Val]: 100%|██████████| 90/90 [00:04<00:00, 21.58it/s, acc=28.6, loss=1.67]


Epoch 19/20 | Train Loss: 1.7647, Train Acc: 27.32% | Val Loss: 1.7495, Val Acc: 28.57%


Epoch 20 [Val]: 100%|██████████| 90/90 [00:05<00:00, 17.59it/s, acc=29.5, loss=1.67]

Epoch 20/20 | Train Loss: 1.7613, Train Acc: 27.41% | Val Loss: 1.7366, Val Acc: 29.45%
Best model saved (val_acc: 29.45%)

Training complete. Best Val Acc: 29.45%


batch,▁▁▂▂▂▂▂▂▃▃▃▄▄▄▄▄▅▅▅▅▅▅▅▅▅▆▆▆▆▆▆▆▆▇▇▇▇▇▇█
batch_loss,▇▃▆▇▇▅▃▆▄▅▇▅▃▅▄▄▃▅▃▁▅▃▄▅▁▅█▄▂▄▆▄▄▆▄▅▆▂▆▄
best_val_acc,▁
epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
lr,████████████▃▃▃▃▁▁▁▁
train_acc,▁▃▃▄▅▆▆▆▇█▅▄▄▅▆▆▆▆▇▇
train_loss,█▅▅▄▃▃▂▂▁▁▃▅▅▃▃▃▂▂▂▁
val_acc,▂▁▁▂▇▃▇▅██▅▁▄▅▆▇▃▅▇█
val_loss,▇▇▆▇▃█▂▃▁▂▅▇▅▅▄▄▇▄▃▂
batch,7538
batch_loss,1.874


In [10]:
for mod in ['models', 'trainer']:
    if mod in sys.modules:
        del sys.modules[mod]

from models import DeepCNNv2
from trainer import train

config = {
    'architecture': 'DeepCNNv2',
    'epochs': 30,
    'lr': 0.001,
    'batch_size': 64,
    'optimizer': 'AdamW',
    'scheduler': 'CosineAnnealing',
    'weight_decay': 1e-4,
}

run = wandb.init(
    project="ML-assn-04",
    name="DeepCNNv2-AdamW-cosine",
    config=config
)

model = DeepCNNv2(num_classes=7).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=config['lr'],
    weight_decay=config['weight_decay']
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=config['epochs'],
    eta_min=1e-6
)

best_model_path = train(
    model, train_loader, val_loader,
    criterion, optimizer, scheduler,
    device, config
)

run.finish()

Epoch 1 [Val]: 100%|██████████| 90/90 [00:03<00:00, 29.68it/s, acc=44.6, loss=1.43]
/content/ml-assn-04/trainer.py:76: UserWarning: The epoch parameter in `scheduler.step()` was not necessary and is being deprecated where possible. Please use `scheduler.step()` to step the scheduler. During the deprecation, if epoch is different from None, the closed form is used instead of the new chainable form, where available. Please open an issue if you are unable to replicate your use case: https://github.com/pytorch/pytorch/issues/new/choose.
  scheduler.step(val_loss)


Epoch 1/30 | Train Loss: 1.6683, Train Acc: 32.72% | Val Loss: 1.4353, Val Acc: 44.64%
Best model saved (val_acc: 44.64%)


Epoch 2 [Val]: 100%|██████████| 90/90 [00:02<00:00, 30.50it/s, acc=45, loss=1.32]


Epoch 2/30 | Train Loss: 1.4253, Train Acc: 44.85% | Val Loss: 1.3930, Val Acc: 44.96%
Best model saved (val_acc: 44.96%)


Epoch 3 [Val]: 100%|██████████| 90/90 [00:02<00:00, 31.06it/s, acc=49.9, loss=1.39]


...

Epoch 29 [Val]: 100%|██████████| 90/90 [00:02<00:00, 30.25it/s, acc=65.2, loss=1.03]


Epoch 29/30 | Train Loss: 0.8380, Train Acc: 69.50% | Val Loss: 0.9471, Val Acc: 65.22%


Epoch 30 [Val]: 100%|██████████| 90/90 [00:02<00:00, 30.03it/s, acc=64.6, loss=0.928]

Epoch 30/30 | Train Loss: 0.8229, Train Acc: 69.76% | Val Loss: 0.9743, Val Acc: 64.64%

Training complete. Best Val Acc: 66.05%


batch,▁▁▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▅▅▅▅▅▆▆▆▆▇▇▇▇▇▇▇██
batch_loss,█▅▅▆▆▄▆▅▄▅▃▄▄▃▄▃▄▄▄▃▄▄▂▃▃▃▄▂▂▃▂▃▂▁▃▁▂▃▁▁
best_val_acc,▁
epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
lr,▁▂▃▄▅▅▅▆▆▅▇▇▇▇▇▆▇█▇▇█████████▇
train_acc,▁▃▄▄▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇███████
train_loss,█▆▅▅▄▄▄▄▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁
val_acc,▁▁▃▄▄▄▅▆▆▅▆▆▇▇▆▆▇█▇▆█▇▇███████
val_loss,█▇▆▅▄▄▄▃▃▄▃▃▂▂▃▃▂▁▂▂▁▂▁▁▁▁▁▁▁▂
batch,11128
batch_loss,0.75725
